# 10 — Descarga de corridas → DriveResuelve los 19 accessions de `data/organismos.tsv` a corridas contra la ENA ybaja los `.sra` a `tesis/80_sra/`.**Las sesiones de Colab se mueren, y eso es lo normal, no la excepción.** Estenotebook está hecho para eso: el estado del trabajo es qué archivos existen enDrive, así que re-ejecutarlo retoma donde quedó. Usá `LIMITE` para que cadasesión haga una tanda y termine.Corré antes `00_setup.ipynb`.

## Preámbulo: montar Drive y clonar el repoEl repo es público, así que el clon no necesita credenciales. **Los notebooksllaman a los scripts del repo en vez de reimplementarlos**: el criterio deselección de corridas y el de verificación de ensamblados tienen que vivir enun solo lugar, o dejan de ser reproducibles.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import os, pathlibDRIVE = pathlib.Path('/content/drive/MyDrive/tesis')CLON  = pathlib.Path('/content/tesis')assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'print('Drive OK:', DRIVE)

In [ ]:
import subprocessif CLON.exists():    print(subprocess.run(['git','-C',str(CLON),'pull','--ff-only'],                         capture_output=True, text=True).stdout)else:    print(subprocess.run(['git','clone','--depth','1','https://github.com/youkonskernel-afk/tesis.git',str(CLON)],                         capture_output=True, text=True).stderr)print(subprocess.run(['git','-C',str(CLON),'log','--oneline','-1'],                     capture_output=True, text=True).stdout)

In [ ]:
import os, glob, shutilSRA_VER = '3.1.1'c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')if c: os.environ['PATH'] = c[0] + ':' + os.environ['PATH']for b in ['prefetch', 'vdb-validate', 'jq', 'curl']:    assert shutil.which(b), f'falta {b} — corré 00_setup.ipynb'SRA = DRIVE / '80_sra'; SRA.mkdir(parents=True, exist_ok=True)STAGING = pathlib.Path('/content/sra_staging'); STAGING.mkdir(exist_ok=True)MANIFIESTO_DRIVE = DRIVE / '00_manifiestos' / 'srr_manifest.tsv'print('destino :', SRA)print('staging :', STAGING, f'({shutil.disk_usage("/content").free/1e9:.0f} GB libres)')

## 1. ManifiestoSe genera con `scripts/fetch_runs.sh manifest`, que consulta la ENA y filtra adatos de RNA: `library_source = TRANSCRIPTOMIC` —el filtro duro, porque variosBioProjects mezclan corridas GENOMIC— y `SINGLE` para RNA-Seq, porque el PAIREDde un proyecto de RNA-Seq no es sRNA-seq.Se guarda una copia en Drive: el clon es efímero y el manifiesto define eltrabajo pendiente.

In [ ]:
import subprocess, shutil as _shif MANIFIESTO_DRIVE.exists():    print('ya hay manifiesto en Drive; lo reuso.')    print('Para regenerarlo, borralo primero.')    _sh.copy(MANIFIESTO_DRIVE, CLON / 'data' / 'srr_manifest.tsv')else:    r = subprocess.run(['./scripts/fetch_runs.sh', 'manifest'], cwd=CLON,                       capture_output=True, text=True)    print(r.stdout[-4000:]); print(r.stderr[-4000:])    MANIFIESTO_DRIVE.parent.mkdir(parents=True, exist_ok=True)    _sh.copy(CLON / 'data' / 'srr_manifest.tsv', MANIFIESTO_DRIVE)    print('copia guardada en', MANIFIESTO_DRIVE)

## 2. Bajar una tanda`prefetch` escribe primero en el disco de la VM, se corre `vdb-validate`, y**recién ahí** se mueve a Drive. Dos motivos: escribir GB directo al FUSE deDrive es lento e inestable, y un `.sra` truncado no falla ruidosamente —alineade menos—, así que el que no valida se descarta y nunca llega al destino.`LIMITE` acota la tanda. Empezá chico para medir cuánto rinde tu sesión.

In [ ]:
LIMITE = 5          # corridas por tandaORGANISMO = ''      # '' = todos; o 'prupe', 'gadmo', ...env = dict(os.environ,           SRA_DEST=str(SRA),           SRA_STAGING=str(STAGING),           SRA_LEDGER=str(CLON / 'data' / 'sra_md5.tsv'),           MANIFEST=str(CLON / 'data' / 'srr_manifest.tsv'))cmd = ['./scripts/fetch_runs.sh', 'prefetch']if ORGANISMO: cmd.append(ORGANISMO)cmd += ['-n', str(LIMITE)]p = subprocess.Popen(cmd, cwd=CLON, env=env, text=True,                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT)for ln in p.stdout:    print(ln, end='')p.wait()

## 3. Guardar el ledgerLos md5 van a git, no solo a Drive. Copiá esta salida a `data/sra_md5.tsv` en elrepo y commiteala.

In [ ]:
led = CLON / 'data' / 'sra_md5.tsv'print(led.read_text() if led.exists() else '(vacío)')

## 4. RepetirVolvé a correr la celda 2 hasta que `90_estado.ipynb` no muestre faltantes.Cada tanda retoma sola.**Colab Free no está pensado para trabajo desatendido largo**: el uso sostenidolleva a throttling. Conviene espaciar las tandas en vez de encadenarlas.